In [8]:
!pip install skyfield pandas matplotlib pytz gradio

In [9]:
from skyfield.api import load, Topos
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import gradio as gr

In [10]:
PRESET_LOCATIONS = {
    "Cairo, Egypt": (30.0444, 31.2357),
    "Beni Suef, Egypt": (29.0661, 31.0994),
    "Alexandria, Egypt": (31.2001, 29.9187),
    "Aswan, Egypt": (24.0889, 32.8998)
}

def utc_to_local(utc_iso_string, tz_name="Africa/Cairo"):
    local_tz = pytz.timezone(tz_name)
    dt_utc = datetime.fromisoformat(utc_iso_string.replace("Z", "+00:00"))
    dt_local = dt_utc.astimezone(local_tz)
    return dt_local.strftime("%Y-%m-%d %H:%M:%S %Z")

def compute_satellite_day_data(satellite, observer, times):
    records = []
    for t in times:
        difference = satellite - observer
        topocentric = difference.at(t)
        alt, az, distance = topocentric.altaz()

        records.append({
            "time_utc": t.utc_iso(),
            "altitude_deg": alt.degrees,
            "azimuth_deg": az.degrees,
            "distance_km": distance.km
        })

    return pd.DataFrame(records)

def extract_passes(df, satellite_name, min_tracking_altitude=10, tz_name="Africa/Cairo"):
    temp = df.copy()
    temp["visible"] = temp["altitude_deg"] > 0
    temp["pass_group"] = (temp["visible"] != temp["visible"].shift()).cumsum()

    passes = []

    for _, group in temp.groupby("pass_group"):
        if group["visible"].iloc[0]:
            pass_start_utc = group.iloc[0]["time_utc"]
            pass_end_utc = group.iloc[-1]["time_utc"]
            duration_minutes = len(group)
            max_altitude_deg = group["altitude_deg"].max()
            avg_altitude_deg = group["altitude_deg"].mean()

            idx_max = group["altitude_deg"].idxmax()
            peak_time_utc = temp.loc[idx_max, "time_utc"]

            useful_points = int((group["altitude_deg"] >= min_tracking_altitude).sum())

            tracking_score = (
                0.4 * max_altitude_deg +
                0.2 * duration_minutes +
                0.2 * avg_altitude_deg +
                0.2 * useful_points
            )

            passes.append({
                "satellite_name": satellite_name,
                "pass_start_utc": pass_start_utc,
                "pass_end_utc": pass_end_utc,
                "pass_start_local": utc_to_local(pass_start_utc, tz_name),
                "pass_end_local": utc_to_local(pass_end_utc, tz_name),
                "peak_time_utc": peak_time_utc,
                "peak_time_local": utc_to_local(peak_time_utc, tz_name),
                "duration_minutes": duration_minutes,
                "max_altitude_deg": max_altitude_deg,
                "avg_altitude_deg": avg_altitude_deg,
                "useful_tracking_points": useful_points,
                "practical_tracking_pass": max_altitude_deg >= min_tracking_altitude,
                "tracking_score": tracking_score
            })

    return pd.DataFrame(passes)

def build_tracking_table(df, pass_start_utc, pass_end_utc, tz_name="Africa/Cairo"):
    tracking_df = df[
        (df["time_utc"] >= pass_start_utc) &
        (df["time_utc"] <= pass_end_utc)
    ].copy()

    tracking_df = tracking_df[["time_utc", "azimuth_deg", "altitude_deg", "distance_km"]].copy()
    tracking_df["time_local"] = tracking_df["time_utc"].apply(lambda x: utc_to_local(x, tz_name))
    tracking_df["delta_azimuth_deg"] = tracking_df["azimuth_deg"].diff().fillna(0)
    tracking_df["delta_altitude_deg"] = tracking_df["altitude_deg"].diff().fillna(0)

    return tracking_df

def convert_degrees_to_steps(tracking_df, azimuth_deg_per_step=1.8, altitude_deg_per_step=1.8):
    commands_df = tracking_df[["time_utc", "time_local", "delta_azimuth_deg", "delta_altitude_deg"]].copy()
    commands_df["azimuth_steps"] = (commands_df["delta_azimuth_deg"] / azimuth_deg_per_step).round().astype(int)
    commands_df["altitude_steps"] = (commands_df["delta_altitude_deg"] / altitude_deg_per_step).round().astype(int)
    return commands_df

def update_coordinates(location_name):
    lat, lon = PRESET_LOCATIONS[location_name]
    return lat, lon, location_name

In [11]:
def run_tracking_dashboard(
    observer_name,
    latitude,
    longitude,
    year,
    month,
    day,
    min_tracking_altitude,
    time_step_minutes,
    max_visual_satellites,
    timezone_name
):
    ts = load.timescale()
    observer = Topos(latitude_degrees=float(latitude), longitude_degrees=float(longitude))

    satellites = []

    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=stations&FORMAT=tle'
    visual_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=visual&FORMAT=tle'

    stations_sats = load.tle_file(stations_url)
    visual_sats = load.tle_file(visual_url)

    satellites.extend(stations_sats)
    satellites.extend(visual_sats[:int(max_visual_satellites)])

    unique_sats = {}
    for sat in satellites:
        unique_sats[sat.name] = sat
    satellites = list(unique_sats.values())

    minutes = list(range(0, 24 * 60, int(time_step_minutes)))
    times = ts.utc(int(year), int(month), int(day), 0, minutes)

    daily_data_by_satellite = {}
    best_pass_per_satellite = []

    for satellite in satellites:
        try:
            day_df = compute_satellite_day_data(satellite, observer, times)
            daily_data_by_satellite[satellite.name] = day_df

            passes_df = extract_passes(
                day_df,
                satellite_name=satellite.name,
                min_tracking_altitude=float(min_tracking_altitude),
                tz_name=timezone_name
            )

            if len(passes_df) > 0:
                practical_passes = passes_df[passes_df["practical_tracking_pass"] == True].copy()

                if len(practical_passes) > 0:
                    best_pass = practical_passes.sort_values(
                        by=["tracking_score", "max_altitude_deg", "duration_minutes"],
                        ascending=[False, False, False]
                    ).iloc[0]
                else:
                    best_pass = passes_df.sort_values(
                        by=["tracking_score", "max_altitude_deg", "duration_minutes"],
                        ascending=[False, False, False]
                    ).iloc[0]

                best_pass_per_satellite.append(best_pass.to_dict())

        except Exception:
            continue

    if len(best_pass_per_satellite) == 0:
        empty_df = pd.DataFrame()
        empty_file = "no_results.csv"
        empty_df.to_csv(empty_file, index=False)
        return (
            "No visible passes found for the selected date and location.",
            empty_df,
            empty_df,
            empty_df,
            None,
            None,
            None,
            empty_file,
            empty_file,
            empty_file
        )

    best_pass_per_satellite_df = pd.DataFrame(best_pass_per_satellite)

    practical_best_df = best_pass_per_satellite_df[
        best_pass_per_satellite_df["practical_tracking_pass"] == True
    ].copy()

    if len(practical_best_df) > 0:
        overall_best = practical_best_df.sort_values(
            by=["tracking_score", "max_altitude_deg", "duration_minutes"],
            ascending=[False, False, False]
        ).iloc[0]
    else:
        overall_best = best_pass_per_satellite_df.sort_values(
            by=["tracking_score", "max_altitude_deg", "duration_minutes"],
            ascending=[False, False, False]
        ).iloc[0]

    best_satellite_name = overall_best["satellite_name"]
    best_day_df = daily_data_by_satellite[best_satellite_name]

    best_tracking_df = build_tracking_table(
        best_day_df,
        overall_best["pass_start_utc"],
        overall_best["pass_end_utc"],
        tz_name=timezone_name
    )

    best_tracking_filtered_df = best_tracking_df[
        best_tracking_df["altitude_deg"] >= float(min_tracking_altitude)
    ].copy()

    if len(best_tracking_filtered_df) == 0:
        best_tracking_filtered_df = best_tracking_df.copy()

    tracker_commands_df = convert_degrees_to_steps(best_tracking_filtered_df)

    summary_text = f"""
Best Satellite: {overall_best['satellite_name']}
Observer: {observer_name}
Pass Start: {overall_best['pass_start_local']}
Pass End: {overall_best['pass_end_local']}
Peak Time: {overall_best['peak_time_local']}
Duration (min): {int(overall_best['duration_minutes'])}
Max Altitude (deg): {round(overall_best['max_altitude_deg'], 2)}
Average Altitude (deg): {round(overall_best['avg_altitude_deg'], 2)}
Useful Tracking Points: {int(overall_best['useful_tracking_points'])}
Tracking Score: {round(overall_best['tracking_score'], 2)}
""".strip()

    best_passes_file = "best_pass_per_satellite.csv"
    tracking_file = "best_tracking_table.csv"
    commands_file = "tracker_motor_commands.csv"

    best_pass_per_satellite_df.to_csv(best_passes_file, index=False)
    best_tracking_filtered_df.to_csv(tracking_file, index=False)
    tracker_commands_df.to_csv(commands_file, index=False)

    fig1, ax1 = plt.subplots(figsize=(10, 5))
    ax1.bar(best_pass_per_satellite_df["satellite_name"], best_pass_per_satellite_df["tracking_score"])
    ax1.set_title("Tracking Score per Satellite")
    ax1.set_xlabel("Satellite")
    ax1.set_ylabel("Tracking Score")
    plt.xticks(rotation=75)
    plt.tight_layout()

    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(best_day_df["altitude_deg"])
    ax2.axhline(y=0, linestyle='--')
    ax2.axhline(y=float(min_tracking_altitude), linestyle='--')
    ax2.set_title(f"Full-Day Altitude Profile for {best_satellite_name}")
    ax2.set_xlabel("Time Step")
    ax2.set_ylabel("Altitude (degrees)")
    plt.tight_layout()

    fig3, ax3 = plt.subplots(figsize=(10, 5))
    ax3.plot(best_tracking_filtered_df["azimuth_deg"], best_tracking_filtered_df["altitude_deg"], marker='o')
    ax3.set_title(f"Tracking Path for {best_satellite_name}")
    ax3.set_xlabel("Azimuth (degrees)")
    ax3.set_ylabel("Altitude (degrees)")
    plt.tight_layout()

    return (
        summary_text,
        best_pass_per_satellite_df,
        best_tracking_filtered_df,
        tracker_commands_df,
        fig1,
        fig2,
        fig3,
        best_passes_file,
        tracking_file,
        commands_file
    )

In [12]:
custom_css = """
body {
    font-family: Arial, sans-serif;
}
.gradio-container {
    max-width: 1200px !important;
}
.main-title {
    text-align: center;
    font-size: 32px;
    font-weight: bold;
    margin-bottom: 8px;
}
.sub-title {
    text-align: center;
    font-size: 16px;
    color: #555;
    margin-bottom: 20px;
}
.section-note {
    font-size: 14px;
    color: #444;
    margin-bottom: 10px;
}
"""

In [13]:
with gr.Blocks(css=custom_css, title="Satellite Tracking Dashboard") as demo:
    gr.HTML('<div class="main-title">Automated Multi-Satellite Tracking Dashboard</div>')
    gr.HTML('<div class="sub-title">Pass prediction, optimal satellite selection, tracking analysis, and motor command generation</div>')

    with gr.Group():
        gr.Markdown("### Configuration")
        gr.Markdown("Choose a preset location or edit the coordinates manually, then run the tracking analysis.")

        with gr.Row():
            location_dropdown = gr.Dropdown(
                choices=list(PRESET_LOCATIONS.keys()),
                value="Cairo, Egypt",
                label="Preset Location"
            )
            observer_name = gr.Textbox(label="Observer Name", value="Cairo, Egypt")

        with gr.Row():
            latitude = gr.Number(label="Latitude", value=30.0444)
            longitude = gr.Number(label="Longitude", value=31.2357)

        with gr.Row():
            year = gr.Number(label="Year", value=2026)
            month = gr.Number(label="Month", value=4)
            day = gr.Number(label="Day", value=17)

        with gr.Row():
            min_tracking_altitude = gr.Slider(label="Minimum Tracking Altitude", minimum=0, maximum=30, step=1, value=10)
            time_step_minutes = gr.Slider(label="Time Step (minutes)", minimum=1, maximum=10, step=1, value=1)
            max_visual_satellites = gr.Slider(label="Max Visual Satellites", minimum=5, maximum=50, step=1, value=20)

        timezone_name = gr.Textbox(label="Timezone", value="Africa/Cairo")

        run_button = gr.Button("Run Tracking Analysis", variant="primary")

    with gr.Tabs():
        with gr.Tab("Summary"):
            summary_output = gr.Textbox(label="Mission Summary", lines=10)

        with gr.Tab("Best Passes"):
            best_passes_output = gr.Dataframe(label="Best Pass Per Satellite")
            best_passes_download = gr.File(label="Download Best Passes CSV")

        with gr.Tab("Tracking Table"):
            tracking_output = gr.Dataframe(label="Best Tracking Table")
            tracking_download = gr.File(label="Download Tracking Table CSV")

        with gr.Tab("Motor Commands"):
            commands_output = gr.Dataframe(label="Tracker Motor Commands")
            commands_download = gr.File(label="Download Commands CSV")

        with gr.Tab("Plots"):
            plot1_output = gr.Plot(label="Tracking Score per Satellite")
            plot2_output = gr.Plot(label="Full-Day Altitude Profile")
            plot3_output = gr.Plot(label="Tracking Path")

    location_dropdown.change(
        fn=update_coordinates,
        inputs=location_dropdown,
        outputs=[latitude, longitude, observer_name]
    )

    run_button.click(
        fn=run_tracking_dashboard,
        inputs=[
            observer_name,
            latitude,
            longitude,
            year,
            month,
            day,
            min_tracking_altitude,
            time_step_minutes,
            max_visual_satellites,
            timezone_name
        ],
        outputs=[
            summary_output,
            best_passes_output,
            tracking_output,
            commands_output,
            plot1_output,
            plot2_output,
            plot3_output,
            best_passes_download,
            tracking_download,
            commands_download
        ]
    )

demo.launch(share=True)

/tmp/ipykernel_627/3911347110.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="Satellite Tracking Dashboard") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://095b9d949cff0b0abc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
